# Fine-Tuning Qwen2.5-1.5B-Instruct — OS Tutor (Unsloth)

**Method:** QLoRA via Unsloth · **GPU:** T4 Free Colab

> ⚠️ **FIRST:** Runtime → Change runtime type → **T4 GPU** → Save

In [ ]:
# ============================================================
# CELL 1 — Install packages
# After this finishes: Runtime > Restart Session > run Cell 2 onward
# ============================================================
import sys
print(f'Python {sys.version}')

!pip install -q unsloth datasets

print('\nInstall complete.')
print('Now: Runtime > Restart Session, then run from Cell 2.')

In [ ]:
# ============================================================
# CELL 2 — Imports & Config
# ============================================================
import os, sys, torch, pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig

print(f'Python  : {sys.version}')
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# ── Model & data paths ────────────────────────────────────────
MODEL_ID          = 'Qwen/Qwen2.5-1.5B-Instruct'
TRAIN_CSV         = 'dataset_v2_train.csv'
VAL_CSV           = 'dataset_v2_val.csv'
OUTPUT_DIR        = './qwen-os-tutor-lora'
MERGE_OUTPUT      = './qwen-os-tutor-merged'
MAX_SEQ_LEN       = 1024

# ── LoRA ──────────────────────────────────────────────────────
LORA_R            = 16
LORA_ALPHA        = 32
LORA_DROPOUT      = 0.05
TARGET_MODULES    = ['q_proj','k_proj','v_proj','o_proj',
                     'gate_proj','up_proj','down_proj']

# ── Training ──────────────────────────────────────────────────
EPOCHS            = 3
BATCH_SIZE        = 2
GRAD_ACCUM        = 8          # effective batch = 16
LR                = 2e-4

# ── Weighted sampling ─────────────────────────────────────────
OVERSAMPLE_CATS   = {'comparative','analytical','negative_edge_case','applied_conceptual'}
OVERSAMPLE_FACTOR = 3

print('\nConfig loaded.')

In [ ]:
# ============================================================
# CELL 3 — Upload CSV files
# Upload: dataset_v2_train.csv  AND  dataset_v2_val.csv
# ============================================================
from google.colab import files

print('Upload dataset_v2_train.csv and dataset_v2_val.csv when the picker opens...')
uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))

train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)

print(f'\nTrain: {train_df.shape}  |  Val: {val_df.shape}')
print(f'Columns: {train_df.columns.tolist()}')
print('\nCategory counts (train):')
print(train_df['category'].value_counts())

In [ ]:
# ============================================================
# CELL 4 — Load Tokenizer + Pre-format Dataset to Text Strings
#
# KEY FIX: We apply the chat template HERE (before loading the model)
# so the dataset contains plain text strings ('text' field).
# SFTTrainer then just reads dataset_text_field='text' — no
# formatting_func needed, which eliminates all Cell 6 errors.
# ============================================================

# Load tokenizer only (CPU, no VRAM cost)
print('Loading tokenizer (CPU only)...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = 'right'
print(f'Tokenizer loaded. Vocab size: {tokenizer.vocab_size}')

# ── Weighted oversampling ─────────────────────────────────────
def apply_weighted_sampling(df, cats, factor):
    base       = df[~df['category'].isin(cats)]
    minority   = df[df['category'].isin(cats)]
    oversampled = pd.concat([minority] * factor, ignore_index=True)
    result = pd.concat([base, oversampled], ignore_index=True).sample(frac=1, random_state=42)
    print(f'\nAfter oversampling: {len(result)} total examples')
    print(result['category'].value_counts())
    return result

# ── Format each row to a text string via chat template ────────
def row_to_text(row):
    messages = [
        {'role': 'system',    'content': str(row['system'])},
        {'role': 'user',      'content': str(row['prompt'])},
        {'role': 'assistant', 'content': str(row['completion'])},
    ]
    # apply_chat_template returns a plain string like:
    # <|im_start|>system\n...<|im_end|>\n<|im_start|>user\n...
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,             # return string, not token IDs
        add_generation_prompt=False # don't add assistant: prefix at end
    )

def df_to_hf(df):
    """Convert dataframe to HuggingFace Dataset with a 'text' field."""
    return Dataset.from_list([
        {'text': row_to_text(r)} for _, r in df.iterrows()
    ])

# Apply oversampling to train set only
train_df_sampled = apply_weighted_sampling(train_df, OVERSAMPLE_CATS, OVERSAMPLE_FACTOR)

print('\nBuilding HuggingFace datasets...')
train_dataset = df_to_hf(train_df_sampled)
val_dataset   = df_to_hf(val_df)

print(f'\nTrain HF dataset : {len(train_dataset)} examples')
print(f'Val   HF dataset : {len(val_dataset)} examples')
print(f'\nSample text (first 300 chars):')
print(train_dataset[0]['text'][:300])

In [ ]:
# ============================================================
# CELL 5 — Load Model in 4-bit + Inject LoRA (Unsloth)
# Runtime: ~3-5 min (downloads ~1.5 GB weights)
# ============================================================

print(f'Loading {MODEL_ID} in 4-bit via Unsloth...')

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = MODEL_ID,
    max_seq_length= MAX_SEQ_LEN,
    load_in_4bit  = True,
    dtype         = None,   # auto (fp16 on T4, bf16 on A100)
)

print('\nInjecting LoRA adapters...')
model = FastLanguageModel.get_peft_model(
    model,
    r                          = LORA_R,
    lora_alpha                 = LORA_ALPHA,
    lora_dropout               = LORA_DROPOUT,
    target_modules             = TARGET_MODULES,
    bias                       = 'none',
    use_gradient_checkpointing = 'unsloth',  # memory-efficient
    random_state               = 42,
)

model.print_trainable_parameters()

alloc = torch.cuda.memory_allocated() / 1e9
resv  = torch.cuda.memory_reserved()  / 1e9
print(f'\nVRAM — Allocated: {alloc:.2f} GB | Reserved: {resv:.2f} GB')
print('\nModel ready!')

In [ ]:
# ============================================================
# CELL 6 — Train with SFTTrainer
# Runtime: ~25-35 min on T4
#
# No formatting_func. Dataset already has 'text' strings from Cell 4.
# SFTTrainer reads them directly via dataset_text_field='text'.
# ============================================================

# Calculate warmup_steps from total training steps
steps_per_epoch  = len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM)
total_steps      = steps_per_epoch * EPOCHS
warmup_steps     = max(1, int(total_steps * 0.05))

print(f'Training plan:')
print(f'  Examples        : {len(train_dataset)}')
print(f'  Effective batch : {BATCH_SIZE * GRAD_ACCUM}')
print(f'  Steps/epoch     : {steps_per_epoch}')
print(f'  Total steps     : {total_steps}')
print(f'  Warmup steps    : {warmup_steps}')
print()

sft_config = SFTConfig(
    # ── Output ───────────────────────────────────────
    output_dir                  = OUTPUT_DIR,

    # ── Schedule ─────────────────────────────────────
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    gradient_checkpointing      = True,

    # ── Optimizer ────────────────────────────────────
    optim                       = 'adamw_8bit',
    learning_rate               = LR,
    lr_scheduler_type           = 'cosine',
    warmup_steps                = warmup_steps,  # avoid deprecation warning
    weight_decay                = 0.01,

    # ── Precision ────────────────────────────────────
    fp16                        = not torch.cuda.is_bf16_supported(),
    bf16                        = torch.cuda.is_bf16_supported(),

    # ── Dataset ──────────────────────────────────────
    dataset_text_field          = 'text',   # plain text field from Cell 4
    max_seq_length              = MAX_SEQ_LEN,
    packing                     = False,    # set True to speed up if examples are short

    # ── Logging & checkpointing ───────────────────────
    logging_steps               = 10,
    eval_strategy               = 'steps',
    eval_steps                  = 50,
    save_strategy               = 'steps',
    save_steps                  = 50,
    save_total_limit            = 2,
    load_best_model_at_end      = True,
    metric_for_best_model       = 'eval_loss',
    greater_is_better           = False,

    # ── Misc ─────────────────────────────────────────
    report_to                   = 'none',
    remove_unused_columns       = True,
)

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    args          = sft_config,
    train_dataset = train_dataset,
    eval_dataset  = val_dataset,
)

print('Starting training...')
result = trainer.train()

print(f'\nDone!')
print(f'  Total steps   : {result.global_step}')
print(f'  Training loss : {result.training_loss:.4f}')

In [ ]:
# ============================================================
# CELL 7 — Save LoRA Adapter + Merge into Full Model
# ============================================================

# Save just the LoRA adapter weights (~50 MB)
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'LoRA adapter saved: {OUTPUT_DIR}')
print('Files:', os.listdir(OUTPUT_DIR))

# Merge adapter back into full fp16 model (~3 GB standalone)
print('\nMerging LoRA into base model...')
model.save_pretrained_merged(
    MERGE_OUTPUT,
    tokenizer,
    save_method = 'merged_16bit',
)
print(f'Merged model saved: {MERGE_OUTPUT}')

# Optional: export as GGUF for Ollama / llama.cpp
# model.save_pretrained_gguf(MERGE_OUTPUT, tokenizer, quantization_method='q4_k_m')

In [ ]:
# ============================================================
# CELL 8 — Quick Inference Test
# ============================================================

# Switch model to fast inference mode
FastLanguageModel.for_inference(model)

SYSTEM_PROMPT = (
    'You are an Operating Systems tutor. Answer ONLY using the exact textbook passages '
    'provided below. Do NOT add your own examples, do NOT invent information. '
    'If the answer is not in the provided text, reply EXACTLY with OUT_OF_CONTEXT and nothing else.'
)

def ask(context: str, question: str) -> str:
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': f'Textbook passages:\n{context}\n\nQuestion: {question}'},
    ]
    # Format prompt using chat template (assistant turn left open)
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors='pt').to('cuda')
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens     = 256,
            temperature        = 0.1,
            do_sample          = True,
            repetition_penalty = 1.1,
            use_cache          = True,
        )
    # Decode only the newly generated tokens
    new_tokens = output_ids[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


ctx = (
    'A deadlock is a situation where a set of processes are blocked because '
    'each process is holding a resource and waiting for another resource '
    'acquired by some other process.'
)

print('=== Test 1: In-context question ===')
print('Q: What is a deadlock?')
print('A:', ask(ctx, 'What is a deadlock?'))

print('\n=== Test 2: Out-of-context (expect OUT_OF_CONTEXT) ===')
print('Q: What is the capital of France?')
print('A:', ask(ctx, 'What is the capital of France?'))

In [ ]:
# ============================================================
# CELL 9 — Download LoRA Adapter as ZIP
# ============================================================
import shutil
from google.colab import files

zip_path = shutil.make_archive('qwen-os-tutor-lora', 'zip', OUTPUT_DIR)
print(f'Zipped: {zip_path}')
files.download(zip_path)
print('Download started!')